# Manipulation et analyse d'image avec `scikit-image`

Dans ce TP, nous allons utiliser quelques unes de fonctionnalités proposée par scikit-image pour la manipulation et l'analyse d'image.

Scikit-image s'applique sur des tableaux numpy, de fait, une bonne connaissance de numpy et une certaine aisance à la manipulation des tableaux numpy est nécessaire pour pouvoir utiliser parfaitement `scikit-image`.

In [ ]:
!wget -O la-havana.jpg https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/AvMalecon-LaHabanaCuba-04735.jpg/1920px-AvMalecon-LaHabanaCuba-04735.jpg

import pathlib
from functools import partial
from typing import Iterable, Optional, Union, Tuple
from PIL import Image, ImageFilter
import numpy as np
import scipy
import matplotlib.pyplot as plt
import skimage
from skimage import color, feature, exposure, transform, filters
from skimage.segmentation import active_contour
from skimage.filters import gaussian
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import ORB


##################################################
### Fonction dédiée à l'affichage (Ignorez la) ###
##################################################
def side_by_side(
    images: Iterable[Union[Image.Image, np.ndarray]],
    title: Optional[str] = None,
    subtitles: Optional[Iterable[str]] = None,
    dpi=150,
) -> None:
    fig, axes = plt.subplots(1, len(images), dpi=dpi)

    for i, (ax, im) in enumerate(zip(axes, images)):
        if type(im) is np.ndarray:
            mode = "gray" if len(im.shape) == 2 else "viridis"
        else:
            mode = "gray" if im.mode == "L" else "viridis"
        ax.imshow(im, cmap=mode)
        ax.axis("off")
        if subtitles:
            ax.set_title(subtitles[i], fontsize=8)

    if title:
        fig.suptitle(title, y=0.85)

    fig.subplots_adjust(wspace=0, hspace=0)
    fig.tight_layout()
    plt.show()


###################################

## Histogramme
Comme vous le savez, l'histogramme d'une image est une information simple à obtenir et qui permet d'effectuer des manipulations sur une image pour adapter son contraste.

Comme pour le TP Pillow, commencez par charger une image dans un tableau numpy. Vous trouverez à la racine du dossier courant, une image `la-havana.jpg`. Chargez celle-ci dans une variable `im`.

In [ ]:
# Votre code ici

À partir de cette image, utilisez la fonction [`skimage.exposure.histogram`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.histogram) pour recupérer et afficher l'histogramme de chaque canal.

In [ ]:
# Votre code ici

Utilisez différentes fonctions de corrections de contraste basées sur l'histogramme :
 * [`skimage.exposure.equalize_hist`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_hist) pour appliquer une correction linéaire
 * [`skimage.exposure.equalize_adapthist`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_adapthist) pour appliquer une correction adaptative
 * [`skimage.exposure.rescale_intensity`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.rescale_intensity) pour appliquer une correction dans un intervale de valeur défini (prenez l'intervalle `(30,220)` pour commencer).

Affichez l'image résultante ainsi que le nouvel histogramme.

In [ ]:
# Votre code ici

### Solution

In [ ]:
im = np.array(
    Image.open(
        "./la-havana.jpg",
    )
)

In [ ]:
def display_hist_gray(im: np.ndarray, f=lambda x: x):
    im_hist, im_val = exposure.histogram(im)
    plt.plot(im_val, f(im_hist))
    plt.show()


def display_hist_color(im: np.ndarray, f=lambda x: x):
    im_hist, im_val = exposure.histogram(im, channel_axis=2)
    for h, c in zip(im_hist, ("r", "g", "b")):
        plt.plot(im_val, f(h), color=c)
    plt.show()


def display_hist(im: np.ndarray, f=lambda x: x):
    if len(im.shape) == 3:
        display_hist_color(im, f)
    else:
        display_hist_gray(im, f)


def display_cumhist(im: np.ndarray):
    display_hist(im, np.cumsum)


display_hist(im)
display_cumhist(im)

In [ ]:
im_eq = exposure.equalize_hist(im)
side_by_side((im, im_eq), subtitles=("Original", "Égalisé"))
display_hist(im_eq)
display_cumhist(im_eq)

In [ ]:
im_adapt = exposure.equalize_adapthist(im)
side_by_side((im, im_adapt), subtitles=("Original", "Égalisation adaptatice"))
display_hist(im_adapt)
display_cumhist(im_adapt)

In [ ]:
im_rescale = exposure.rescale_intensity(im, (30, 220))
side_by_side((im, im_rescale), subtitles=("Original", "Échelonnage intensité"))
display_hist(im_rescale)
display_cumhist(im_rescale)

## Identification de caractéristiques

Nous allons utiliser une méthode de detection de caractéristiques entre deux images.

Pour commencer, appliquez une rotation de 45° sur votre image pour obtenir une nouvelle image `im_rotate` en utilisant [`skimage.transform.rotate`](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.rotate)

Utilisez un modèle de détection de caratéristique tel que [`ORB`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB) (**O**riented FAST and **R**otated **B**RIEF) :
 * Créez un objet à partir de la classe `ORB`
 * Appelez la methode `detect_and_extract` de votre objet sur votre image en _**noir et blanc**_
 * À partir de votre objet, vous pourrez récuperer les attributs `keypoints` et `descriptors`

Les `keypoints` sont les coordonnées des caractéristiques extraites sur votre image. Plottez les sur votre image.

In [ ]:
# Votre code ici

### Solution

In [ ]:
im_rotate = transform.rotate(im, 45)
side_by_side((im, im_rotate), subtitles=("Original", "Rotation 45°"))

In [ ]:
# Création du modèle
feature_detector = ORB()

# Recherche des features sur l'image NB
feature_detector.detect_and_extract(color.rgb2gray(im))

# Récupération des keypoints et descriptors calculés
kp1 = feature_detector.keypoints
dsc1 = feature_detector.descriptors

# Affichage
plt.imshow(im)
plt.scatter(kp1[:, 1], kp1[:, 0])

In [ ]:
# Recherche des features sur l'image NB
feature_detector.detect_and_extract(color.rgb2gray(im_rotate))

# Récupération des keypoints et descriptors calculés
kp2 = feature_detector.keypoints
dsc2 = feature_detector.descriptors

# Affichage
plt.imshow(im_rotate)
plt.scatter(kp2[:, 1], kp2[:, 0])

## Alignement de caractéristiques

Utilisez la fonction [`skimage.feature.match_descriptors`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors) pour trouver les caractéristiques communes entre deux images. Vous récupérerez une liste de couple des index de points associés entre eux.

Utilisez la fonction [`skimage.feature.plot_matches`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.plot_matches) pour ploter les alignements de caractéristiques entre le deux images.

In [ ]:
# Votre code ici

### Solution

In [ ]:
aligned_kp = feature.match_descriptors(dsc1, dsc2)

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matches(ax, im, im_rotate, kp1, kp2, aligned_kp)
ax.axis("off")

## Alignement d'images consécutives

### Division de l'image
Utiliser la fonction `split_in_two` ci-dessous pour diviser votre image en deux. Les deux images résulantes auront une section en commun.

In [ ]:
def split_in_two(im: np.ndarray, split_value=None) -> Tuple[np.ndarray, np.ndarray]:
    if not split_value:
        split_value = im.shape[1] // 4
    return im[:, :-split_value, :], im[:, split_value:, :]


# Votre code ici

#### Solution

In [ ]:
def split_in_two(im: np.ndarray, split_value=None) -> Tuple[np.ndarray, np.ndarray]:
    if not split_value:
        split_value = im.shape[1] // 4
    return im[:, :-split_value, :], im[:, split_value:, :]


im_1, im_2 = split_in_two(
    im,
)
side_by_side((im_1, im_2))

### Extraction des keypoints et descripteurs
Nous allons à présent utiliser l'extracteur de caractéristiques ORB pour trouver les caractéristiques communes entre les images comme dans l'exercice précédent.

Stockez les keypoints et les descriptors de chaque image et utilisez la fonction [`skimage.feature.match_descriptors`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors).

**Utilisez le paramètre `max_ratio=0.8`. Cela élimine les keypoints les moins ressemblants.**

Le résultat du `match_descriptor` est la liste des pairs d'indices des keypoints associées.


In [ ]:
# Votre code ici

#### Solution

In [ ]:
feature_detector = ORB()
feature_detector.detect_and_extract(color.rgb2gray(im_1))
kp1 = feature_detector.keypoints
dsc1 = feature_detector.descriptors


feature_detector.detect_and_extract(color.rgb2gray(im_2))
kp2 = feature_detector.keypoints
dsc2 = feature_detector.descriptors

# Recherche des features similaries entre deux images
# Renvoie une liste de deux indices (_,2)
aligned_kp = feature.match_descriptors(dsc1, dsc2, metric="hamming", max_ratio=0.8)

In [ ]:
from skimage import feature

fig, ax = plt.subplots(dpi=200)
feature.plot_matches(ax, im_1, im_2, kp1, kp2, aligned_kp)
ax.axis("off")

### Calcule de la zone commune
 * Calculez la distance moyenne entre les paires de keypoints
 * Calculez la taille de la zone partagée dans les deux images : $$taille_\_zone = taille\_image - distance\_keypoints$$
 * La taille de votre zone partagée étant un nombre de pixels, pensez à transformer la taille de la zone en `int` avec [`numpy.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)


In [ ]:
# Votre code ici

#### Solution

In [ ]:
_, kp_dist = np.abs(kp2[aligned_kp[:, 1]] - kp1[aligned_kp[:, 0]]).mean(axis=0)
shared_size = (im_1.shape[1] - kp_dist).astype(int)

### Fusion des images
* Fusionnez les deux images. Nous allons faire ça simplement :
  * Supprimez la partie gauche de l'image de droite de la taille de la zone partagée
  * Concaténez la première image et le reste de la seconde image avec [`np.concatenate`](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html).
* Affichez le résultat

In [ ]:
# Votre code ici

#### Solution

In [ ]:
plt.figure(dpi=100)
im_fusion = np.concatenate((im_1, im_2[:, shared_size:, :]), axis=1)
plt.imshow(im_fusion)
plt.axis("off")
plt.show()

Fusion utilisant un mélange des deux images plutôt qu'un cut

In [ ]:
def merging_fusion(
    im1: np.ndarray, im2: np.ndarray, merging_size: int, nb_pixel_fusion=30
) -> np.ndarray:
    im_1_left = im1[:, :-merging_size, :]
    im_1_right = im1[:, -merging_size:, :]
    im_2_left = im2[:, :merging_size]
    im_2_right = im2[:, merging_size:]
    merging_ratio = np.zeros(merging_size)
    half_point = merging_size // 2
    merging_ratio[half_point:] = 1
    merging_ratio[half_point - nb_pixel_fusion : half_point + nb_pixel_fusion] = (
        scipy.special.expit(np.linspace(-1, 1, num=nb_pixel_fusion * 2))
    )
    merging_ratio = np.expand_dims(merging_ratio, axis=(0, 2))
    im_center = im_1_right * (1 - merging_ratio) + im_2_left * merging_ratio
    return np.concatenate((im_1_left, im_center, im_2_right), axis=1).astype(np.uint8)


plt.figure(dpi=200)
plt.imshow(merging_fusion(im_1, im_2, shared_size))
plt.axis("off")

### Métrique de comparaison d'image
À partir de votre image :
* Transformez la représentation de votre image en valeur flottants entre 0 et 1 et stockez là dans `im_float`. Vous pouvez utiliser [`skimage.img_as_float`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.img_as_float) pour ça ou le faire vous même
* Créez une image bruitée `im_noise` en ajoutez une image de bruit gaussien de même taille que votre image en utilisant [`np.random.normal`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.normal.html). Vous aurez besoin de [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html#numpy.clip) pour remettre toutes les valeurs entre 0 et 1.
* Créez une image `im_constant` qui correspond à votre image d'origine auquel vous aurez ajouté une constante. Utilisez [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html#numpy.clip) si nécessaire
* Créez une image `im_miroir` qui correspond à l'image miroir sur l'axe horizontal

Comparez ensuite votre image `im_float` avec `im_noise`, `im_constant` et `im_miroir` avec les métriques :
* [`skimage.metrics.structural_similarity`](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) : métrique spécifique à l'image sensée correspondre à notre perception. Valeurs comprises dans $[0,1]$ ; optimale en $1$
* [`skimage.metrics.mean_squared_error`](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.mean_squared_error) : Erreur quadratique moyenne comprise dans $[0, + \infty[$ ; optimale en $0$

In [ ]:
# Votre code ici
# im_float = ???
# im_noise = ???
# im_constant = ???
# im_miroir = ???

### Solution

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error

noise = np.random.normal(0, 0.2, im.shape)

im_float = skimage.img_as_float(im)
mse_float = mean_squared_error(im_float, im_float)
ssim_float = ssim(im_float, im_float, channel_axis=2)

im_noise = np.clip(im_float + noise, 0, 1)
mse_noise = mean_squared_error(im_float, im_noise)
ssim_noise = ssim(im_float, im_noise, channel_axis=2)

im_constant = np.clip(im_float + 0.2, 0, 1)
mse_constant = mean_squared_error(im_float, im_constant)
ssim_constant = ssim(im_float, im_constant, channel_axis=2)

im_miroir = im_float[:, ::-1]
mse_miroir = mean_squared_error(im_float, im_miroir)
ssim_miroir = ssim(im_float, im_miroir, channel_axis=2)

# Mesures
metric_float = f"MSE: {mse_float:.2f}, SSIM: {ssim_float:.2f}"
metric_noise = f"MSE: {mse_noise:.2f}, SSIM: {ssim_noise:.2f}"
metric_constant = f"MSE: {mse_constant:.2f}, SSIM: {ssim_constant:.2f}"
metric_miroir = f"MSE: {mse_miroir:.2f}, SSIM: {ssim_miroir:.2f}"


side_by_side(
    (im_float, im_noise, im_constant, im_miroir),
    subtitles=(metric_float, metric_noise, metric_constant, metric_miroir),
    dpi=200,
)

## Contourage


Exemple de contourage avec zone de support

In [ ]:
from skimage.segmentation import active_contour
from skimage.filters import gaussian
from skimage.color import rgb2gray

POSITION_ELIPSE = (950, 770)
HAUTEUR_ELIPSE = 300
LARGEUR_ELIPSE = 400

s = np.linspace(0, 2 * np.pi, 400)
r = POSITION_ELIPSE[0] + HAUTEUR_ELIPSE * np.sin(s)
c = POSITION_ELIPSE[1] + LARGEUR_ELIPSE * np.cos(s)
init = np.array([r, c]).T

im_bw = (rgb2gray(im) * 255).astype(np.uint8)

snake = active_contour(
    gaussian(im_bw, 3),
    init,
    alpha=0.020,
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(im, cmap=plt.cm.gray)
ax.plot(init[:, 1], init[:, 0], "--r", lw=3)
ax.plot(snake[:, 1], snake[:, 0], "-b", lw=3)
ax.set_xticks([]), ax.set_yticks([])
ax.axis([0, im.shape[1], im.shape[0], 0])

plt.show()